### <h4 style="color:blue;">01 dependencies</h5>

In [12]:
# import libraries

# basic
import os
import random
import pandas as pd
import seaborn as sns
from PIL import Image
from collections import Counter

# visual
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import torch

# <h5 style="color:blue;">

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.9 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.9 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.9 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.9 MB ? eta -:--:--
   ----- -------------------------

In [13]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device yang digunakan: {device}")

if device.type == 'cuda':
    print(f"Nama GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU tidak terdeteksi")

Device yang digunakan: cpu
GPU tidak terdeteksi


In [14]:
import torch
print(torch.cuda.is_available())

False


In [5]:
# base path
TRAIN_PATH = "../data/train"
TEST_PATH  = "../data/test"
CROP_PATH  = "../data/crop"

### <h4 style="color:blue;">02 cropping image</h5>

In [6]:
# coba pake autocrop

# pip install autocrop
# pip install opencv-python-headless
from PIL import Image
from autocrop import Cropper

cropper = Cropper()

cropped_array = cropper.crop('../data/train/realperson/real_002.jpg')

if cropped_array is not None:
    cropped_image = Image.fromarray(cropped_array)
    cropped_image.save('../data/cropped/cropped.png')
    print("Face detected & saved!")
else:
    print("No face detected")

Face detected & saved!


hasilnya autocrop jelek, gabisa lebih dekat lagi

sebelum di-crop

![Before](../data/train/realperson/real_001.jpg)

sesudah di-crop

![After](models/compare2.png)

jadi untuk proses cropping pake scipts python, pakai ROI

### <h4 style="color:blue;">03 cropping image based on csv</h5>

In [10]:
from facenet_pytorch import MTCNN

# Inisialisasi MTCNN
mtcnn = MTCNN(
    keep_all=False,       
    select_largest=True,  
    margin=60,            
    post_process=False,   
    device=device
)

ModuleNotFoundError: No module named 'facenet_pytorch'

In [ ]:
import os
import cv2
import pandas as pd

# Path
csv_path = "../FaceBoundingBox/train/_annotations.csv"
image_folder = "../FaceBoundingBox/train"  # folder gambar asli
output_folder = "../FaceBoundingBox/output"

# Load CSV
df = pd.read_csv(csv_path)

print("Processing images...")
print(f"Total images to process: {len(df)}")

# Counter untuk penamaan
counters = {}

# Loop semua data
for index, row in df.iterrows():
    filename = row['filename']
    label = row['class']
    
    xmin = int(row['xmin'])
    ymin = int(row['ymin'])
    xmax = int(row['xmax'])
    ymax = int(row['ymax'])

    img_path = os.path.join(image_folder, filename)
    
    # Skip kalau file ga ada
    if not os.path.exists(img_path):
        print(f"File tidak ditemukan: {img_path}")
        continue

    # Baca gambar
    img = cv2.imread(img_path)

    # Crop
    cropped = img[ymin:ymax, xmin:xmax]

    # Buat folder kelas kalau belum ada
    class_folder = os.path.join(output_folder, label)
    os.makedirs(class_folder, exist_ok=True)

    # Update counter
    if label not in counters:
        counters[label] = 1

    # Format nama file
    new_name = f"{label}_{counters[label]:03d}.jpg"
    save_path = os.path.join(class_folder, new_name)

    # Simpan hasil crop
    cv2.imwrite(save_path, cropped)

    counters[label] += 1

print("Selesai semua 🚀")